# Second model family: LLaVA-NeXT on FERMAT

Tenth notebook. Everything so far (perception 0.835, reasoning's
`has_error=1` stratum confirmed at 0.801-0.854) is Qwen2.5-VL, 3B and 7B --
one model family. This is the single biggest gap for a paper: is the
result Qwen-specific, or does it hold for a different architecture? This
notebook re-runs both headline measurements -- perception AUROC and the
`has_error=1` stratified reasoning AUROC -- on **LLaVA-NeXT**
(`llava-hf/llava-v1.6-mistral-7b-hf`, ~7B, roughly comparable in scale to
the Qwen2.5-VL-7B reference point) on the identical `n=300` balanced
sample every prior reference run used, so the comparison is apples-to-apples.

## What's reused unchanged from notebook 03

`pilot.data.load_fermat_balanced`, the checkpoint/resume + batch-backoff-
ladder pattern, `pilot.canonicalize.canonical_answer_label`,
`pilot.entropy.cluster_entropy`/`majority_cluster`,
`pilot.parsing.parse_transcription`/`parse_grading`,
`pilot.plotting.bootstrap_auroc_ci`/`stratified_auroc`, and the save-cell
pattern (Drive-first, then repo + push, with `git config` identity set
explicitly).

## What's new: the LLaVA-NeXT adapter

Two things are genuinely model-specific and cannot be reused as-is:

1. **Message shape.** Qwen's messages use a separate `system` role
   (`pilot.prompts.build_messages`). LLaVA-NeXT checkpoints vary by base
   LLM (Mistral, Vicuna, Llama3) and the public chat-template examples for
   this model never show a `system` role being used -- rather than gamble
   on template support that may differ per checkpoint, the system and user
   prompt text are folded into a single `user`-role turn here. This is a
   deliberate, safe design choice, not a placeholder to fix later.
2. **Generation call.** Qwen's pipeline is
   `apply_chat_template(tokenize=False)` -> `qwen_vl_utils.process_vision_info`
   -> `processor(text=..., images=..., ...)` (3 steps, Qwen-specific
   vision-info extraction). LLaVA-NeXT (`transformers` docs, verified
   2026-08-06) supports a single combined call:
   `processor.apply_chat_template(messages, tokenize=True, return_dict=True,
   return_tensors="pt", add_generation_prompt=True)`, which returns
   model-ready inputs (`input_ids`, `pixel_values`, etc.) directly -- no
   separate vision-info utility needed.

**Not yet empirically verified (flagged, not assumed away):** whether
embedding a live PIL image object directly in the message content
(`{"type": "image", "image": <PIL.Image>}`, the same shape
`pilot.prompts.build_messages` already produces) works with this combined
call for this specific checkpoint, the same way it does for Qwen. The
documented public examples use either a `url` key or a bare `{"type":
"image"}` placeholder with the image passed separately to `processor()`.
**Cell 5 (the adapter cell) is a required pre-flight check against the
real loaded model on a single real image before the full sampling loop in
cell 6 runs** -- if direct embedding does not work, the documented
fallback (bare placeholder + image passed positionally to `processor()`)
is a small, known-working change to make there, not a redesign.

Also applied, per the `transformers` LLaVA-NeXT docs' own recommendation:
`processor.patch_size`, `processor.num_additional_image_tokens`, and
`processor.vision_feature_select_strategy` are set explicitly from
`model.config` before generation (recent `transformers` versions warn and
can silently mis-expand image tokens otherwise), and
`processor.tokenizer.padding_side = "left"` for batched generation.

**Prerequisite for running:** cells 2-3 (install, auth) must run this
session.


In [1]:
# Install cell: GPU-dependent packages only.
%pip install -q transformers accelerate datasets huggingface_hub bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 70.2 MB/s eta 0:00:00:00:0100:01


In [2]:
# Auth & code/results access cell. Identical to notebooks 06/08/09 -- reuses
# the HF/GitHub tokens already cached on Drive from prior sessions.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy
import pilot.canonicalize
import pilot.plotting

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")


Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 11.3 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot package imported from: /content/repo/pilot


In [3]:
# Model load cell. LLaVA-NeXT, ~7B (roughly comparable scale to the
# Qwen2.5-VL-7B reference point). Falls back to 4-bit quantization if full
# precision does not fit -- recorded loudly via QUANTIZED, same convention
# as notebooks 05/06/09, since a quantized model is a genuinely different
# measurement, not a transparent substitute.
import torch
from transformers import AutoProcessor, LlavaNextForConditionalGeneration

MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"
QUANTIZED = False

try:
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

# Per the transformers LLaVA-NeXT docs: set these explicitly so image-token
# expansion is computed correctly rather than warned about (or silently
# wrong) on newer transformers versions.
processor.patch_size = model.config.vision_config.patch_size
processor.vision_feature_select_strategy = model.config.vision_feature_select_strategy
processor.num_additional_image_tokens = 1  # CLIP vision tower adds a CLS token
processor.tokenizer.padding_side = "left"  # required for correct batched generation

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")


config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

GPU: NVIDIA A100-SXM4-40GB (39.5 GiB), quantized=False


In [4]:
# Sample cell. Identical to notebook 03 -- same function, seed, and balance,
# so this run is directly comparable to every existing 3B/7B reference run.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5

sample = pilot.data.load_fermat_balanced(
    n=N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC
)
N = len(sample)
n_error = sum(bool(x) for x in sample["has_error"])
print(f"{N} items, {n_error} with a mistake, {N - n_error} clean "
      f"({n_error / N:.0%} error rate)")


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

data/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

...

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]

300 items, 150 with a mistake, 150 clean (50% error rate)


In [5]:
# LLaVA-NeXT adapter cell -- REQUIRED pre-flight check before cell 6 runs.
#
# Builds one real grading message for the first sample item and runs it all
# the way through generate() + decode, so the two genuinely new pieces of
# code (message shape, combined apply_chat_template call) are verified
# against the real loaded model on real data before spending any time on
# the full 300-item loop. If this cell's assertions fail, fix the adapter
# here -- do not proceed to cell 6 with an unverified adapter.
import pilot.prompts


def build_llava_messages(system_prompt: str, user_prompt: str, image) -> list[dict]:
    """Fold system+user into one user turn -- see the intro cell for why:
    LLaVA-NeXT's public chat-template examples never demonstrate a system
    role, and it varies by base LLM (Mistral/Vicuna/Llama3), so this avoids
    depending on template support that may not be there for every
    checkpoint."""
    combined_text = f"{system_prompt}\n\n{user_prompt}"
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": combined_text},
            ],
        },
    ]


def build_llava_grading_messages(image):
    return build_llava_messages(
        pilot.prompts.GRADING_SYSTEM_PROMPT, pilot.prompts.GRADING_USER_PROMPT, image
    )


def build_llava_transcription_messages(image):
    return build_llava_messages(
        pilot.prompts.TRANSCRIPTION_SYSTEM_PROMPT, pilot.prompts.TRANSCRIPTION_USER_PROMPT, image
    )


_test_item = sample[0]
_test_messages = build_llava_grading_messages(_test_item["image"])

try:
    _test_inputs = processor.apply_chat_template(
        _test_messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)
except Exception as exc:
    raise RuntimeError(
        "Embedding a PIL image directly in the message content "
        "({'type': 'image', 'image': <PIL.Image>}) failed for this "
        "checkpoint's chat template. Documented fallback: use a bare "
        "{'type': 'image'} placeholder in the message and pass the image "
        "positionally instead: "
        "processor.apply_chat_template(messages, tokenize=False, "
        "add_generation_prompt=True) then "
        "processor(image, prompt_string, return_tensors='pt')."
    ) from exc

assert "pixel_values" in _test_inputs, (
    "apply_chat_template did not produce pixel_values -- the image was not "
    "recognized. See the RuntimeError message above for the fallback path."
)

with torch.no_grad():
    _test_output = model.generate(**_test_inputs, max_new_tokens=64, do_sample=False)
_test_trimmed = _test_output[:, _test_inputs["input_ids"].shape[1]:]
_test_text = processor.batch_decode(_test_trimmed, skip_special_tokens=True)[0]

print("Adapter pre-flight check OK. Sample output (greedy, 64 tokens):")
print(_test_text)
assert len(_test_text.strip()) > 0, "Model produced empty output -- adapter is broken."


Adapter pre-flight check OK. Sample output (greedy, 64 tokens):
**Reasoning:** The handwritten math problem in the image is a repetition exercise, where the task is to fill in the blanks with the correct number of repetitions. The Answer provided is "4" for the first blank and "4" for the second blank. The student has filled in the same number


In [6]:
# Grading + transcription generation, K=5 each -- mirrors notebook 03's
# structure (this notebook needs both arms, not grading-only). Same
# checkpoint/resume pattern and batch-backoff ladder as every prior
# notebook; only the generation primitives (below) are LLaVA-specific.
import gc
import json
import os
import time

from tqdm.auto import tqdm

K_TRANSCRIPTION = 5
K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True,
        return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_k(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
checkpoint_path = (f"{CHECKPOINT_DIR}/scaleup_{model_slug}_n{N}_seed{SEED}"
                   f"_bal{int(TARGET_ERROR_FRAC * 100)}_kt{K_TRANSCRIPTION}_kg{K_GRADING}"
                   f"{'_4bit' if QUANTIZED else ''}.jsonl")

raw_results = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path) as f:
        raw_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(raw_results[:N]):
        item = sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if (len(entry.get("transcription_samples_raw", [])) != K_TRANSCRIPTION
                or len(entry.get("grading_samples_raw", [])) != K_GRADING):
            break
        valid.append(entry)
    if len(valid) != len(raw_results):
        with open(checkpoint_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
        print(f"Truncated checkpoint from {len(raw_results)} to {len(valid)} valid items.")
    raw_results = valid
    print(f"Resuming from {len(raw_results)} completed items")

if len(raw_results) >= N:
    print(f"All {N} items already done.")
else:
    print(f"Starting from item {len(raw_results) + 1}/{N} "
          f"({N - len(raw_results)} remaining)", flush=True)
    with tqdm(total=(N - len(raw_results)) * (K_TRANSCRIPTION + K_GRADING),
              desc="generating", unit="sample") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < len(raw_results):
                continue
            _t0 = time.time()
            transcription_msgs = build_llava_transcription_messages(item["image"])
            grading_msgs = build_llava_grading_messages(item["image"])
            transcription_texts = generate_k(transcription_msgs, K_TRANSCRIPTION, TEMP)
            pbar.update(K_TRANSCRIPTION)
            grading_texts = generate_k(grading_msgs, K_GRADING, TEMP)
            pbar.update(K_GRADING)
            _elapsed = time.time() - _t0
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "transcription_samples_raw": transcription_texts,
                "grading_samples_raw": grading_texts,
                "quantized": QUANTIZED,
                "elapsed_seconds": _elapsed,
            }
            raw_results.append(entry)
            with open(checkpoint_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            print(f"  item {item_idx + 1}/{N}: {_elapsed:.1f}s "
                  f"({len(raw_results)}/{N} done)", flush=True)

print(f"raw_results: {len(raw_results)} items")


Starting from item 1/300 (300 remaining)


generating:   0%|          | 0/3000 [00:00<?, ?sample/s]

  item 1/300: 30.1s (1/300 done)
  item 2/300: 33.9s (2/300 done)
  item 3/300: 31.3s (3/300 done)
  item 4/300: 31.4s (4/300 done)
  item 5/300: 31.8s (5/300 done)
  item 6/300: 27.3s (6/300 done)
  item 7/300: 33.1s (7/300 done)
  item 8/300: 32.6s (8/300 done)
  item 9/300: 25.1s (9/300 done)
  item 10/300: 32.9s (10/300 done)
  item 11/300: 29.3s (11/300 done)
  item 12/300: 33.3s (12/300 done)
  item 13/300: 34.3s (13/300 done)
  item 14/300: 34.4s (14/300 done)
  item 15/300: 31.3s (15/300 done)
  item 16/300: 42.9s (16/300 done)
  item 17/300: 37.2s (17/300 done)
  item 18/300: 24.2s (18/300 done)
  item 19/300: 28.5s (19/300 done)
  item 20/300: 26.2s (20/300 done)
  item 21/300: 43.3s (21/300 done)
  item 22/300: 38.9s (22/300 done)
  item 23/300: 27.4s (23/300 done)
  item 24/300: 18.7s (24/300 done)
  item 25/300: 26.8s (25/300 done)
  item 26/300: 35.7s (26/300 done)
  item 27/300: 43.4s (27/300 done)
  item 28/300: 32.1s (28/300 done)
  item 29/300: 30.5s (29/300 done)
  i

In [7]:
# Scoring cell. Identical logic to notebook 03's -- same functions, same
# canonicalization path, so LLaVA-NeXT's numbers are computed exactly the
# way every Qwen result was, not a parallel scoring implementation that
# could silently diverge.
import importlib

import pilot.canonicalize
import pilot.entropy
import pilot.parsing

for m in (pilot.parsing, pilot.canonicalize, pilot.entropy):
    importlib.reload(m)

import sympy

_LATEX_OK = pilot.canonicalize.warn_if_latex_parser_missing()
_SYMPY_VERSION = sympy.__version__
print(f"LaTeX parser available: {_LATEX_OK} (sympy {_SYMPY_VERSION})")

scored_results = []
for entry in raw_results:
    item = entry["item"]

    transcription_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_samples_raw"]
    ]
    grading_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]

    transcription_answers = [
        pilot.canonicalize.canonical_answer_label(t) for t in transcription_parsed
    ]

    perception_entropy = pilot.entropy.cluster_entropy(transcription_answers)
    reasoning_entropy = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_parsed]
    )

    majority_transcription, _ = pilot.entropy.majority_cluster(transcription_answers)
    ground_truth_answer = pilot.canonicalize.canonical_answer_label(item["pert_a"])
    transcription_correct = majority_transcription == ground_truth_answer

    majority_grading, _ = pilot.entropy.majority_cluster(
        [None if d is None else str(d) for d in grading_parsed]
    )
    grading_correct = majority_grading in {"0", "1"} and int(majority_grading) == int(
        item["has_error"]
    )

    scored_results.append({
        "orig_q": item["orig_q"],
        "pert_a": item["pert_a"],
        "has_error": item["has_error"],
        "handwriting_style": item["handwriting_style"],
        "image_quality": item["image_quality"],
        "perception_entropy": perception_entropy,
        "reasoning_entropy": reasoning_entropy,
        "transcription_correct": transcription_correct,
        "grading_correct": grading_correct,
        "n_transcription_parse_failures": sum(1 for t in transcription_parsed if t is None),
        "n_grading_parse_failures": sum(1 for d in grading_parsed if d is None),
        "all_transcription_samples_raw": entry["transcription_samples_raw"],
        "all_grading_samples_raw": entry["grading_samples_raw"],
        "model_id": MODEL_ID,
        "quantized": entry["quantized"],
        "n_items": N,
        "k_transcription": K_TRANSCRIPTION,
        "k_grading": K_GRADING,
        "target_error_frac": TARGET_ERROR_FRAC,
        "latex_parser_available": _LATEX_OK,
        "sympy_version": _SYMPY_VERSION,
    })

print(f"Scored {len(scored_results)} items.")

import pandas as pd

_df = pd.DataFrame(scored_results)
accuracy_transcription = _df["transcription_correct"].mean()
accuracy_grading = _df["grading_correct"].mean()
print(f"Transcription accuracy: {accuracy_transcription:.1%}")
print(f"Grading accuracy: {accuracy_grading:.1%} (baseline 50%)")

perception_r = pilot.plotting.bootstrap_auroc_ci(
    _df, "perception_entropy", "transcription_correct", n_boot=10000, seed=0
)
print(f"Perception AUROC: {perception_r['auroc']:.3f} "
      f"[{perception_r['ci_low']:.3f}, {perception_r['ci_high']:.3f}]")

strat = pilot.plotting.stratified_auroc(
    _df, "reasoning_entropy", "grading_correct", "has_error", n_boot=10000, seed=0
)
for level, s in strat["strata"].items():
    minority = min(s["n_error"], s["n_correct"])
    powered = minority >= pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
    print(f"  has_error={level}  n={s['n_items']:3d}  n_wrong={s['n_error']:3d}  "
          f"AUROC {s['auroc']:.3f} [{s['ci_low']:.3f}, {s['ci_high']:.3f}]  "
          f"minority={minority}  {'POWERED' if powered else 'still underpowered'}")
print(f"  sign_reversal      : {strat['sign_reversal']}")
print(f"  pooled_understates : {strat['pooled_understates']}")
print()
print("Qwen2.5-VL-7B reference (report S5.3/S5.4/S7.4): perception 0.835 [0.787, 0.879];")
print("  has_error=1 stratum 0.801-0.834 (confirmed, model-size-independent across 3B/7B).")


LaTeX parser available: True (sympy 1.14.0)
Scored 300 items.
Transcription accuracy: 3.0%
Grading accuracy: 50.0% (baseline 50%)
Perception AUROC: 0.710 [0.525, 0.886]
  has_error=False  n=150  n_wrong=136  AUROC 0.283 [0.178, 0.398]  minority=14  still underpowered
  has_error=True  n=150  n_wrong= 14  AUROC 0.766 [0.628, 0.883]  minority=14  still underpowered
  sign_reversal      : True
  pooled_understates : True

Qwen2.5-VL-7B reference (report S5.3/S5.4/S7.4): perception 0.835 [0.787, 0.879];
  has_error=1 stratum 0.801-0.834 (confirmed, model-size-independent across 3B/7B).


In [8]:
# Save cell: CSV to Drive first, then repo + push. Distinct filename --
# never overwrites any Qwen results.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd

df = pd.DataFrame(scored_results)

model_slug_lower = MODEL_ID.split("/")[-1].lower().replace(".", "")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = (f"scaleup_n{N}_bal{int(TARGET_ERROR_FRAC * 100)}_"
            f"{'4bit_' if QUANTIZED else ''}{model_slug_lower}_{timestamp}.csv")

drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
df.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add LLaVA-NeXT n=300 scale-up results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results to the repo.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")


Backup written to /content/drive/MyDrive/uncertainty-math-vlm/results/scaleup_n300_bal50_llava-v16-mistral-7b-hf_20260806T231143Z.csv
Wrote repo/results/scaleup_n300_bal50_llava-v16-mistral-7b-hf_20260806T231143Z.csv (300 rows)
remote: Permission to sepehrmaleki369/uncertainty-math-vlm.git denied to sepehrmaleki369.
fatal: unable to access 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git/': The requested URL returned error: 403
Push failed (see above). The CSV is safe on Drive and in repo/results/ -- retry the push without re-running the model.
